# Module 17 — Network Flow and Matching

## What you will discover

Every cell below runs this module's **real** problem-bank solutions and asserts
their behaviour. Nothing here prints a claim it has not computed.

The assertions are lifted directly from `problems/tests/`, so they cannot drift
from the implementations — if a signature changes, the tests break first.

**One cell near the end is deliberately broken.** Fixing it is the exercise.

## Setup

The solutions directory goes on `sys.path` relative to this notebook's own
location. Never hard-code an absolute path — `tools/check_links.py` fails the
build on them, because a path with a username in it works on exactly one
machine.

In [ ]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "problems" / "solutions"))

import itertools
import random
from p01_max_flow_value import max_flow_value
from p02_min_cut_edges import min_cut_edges
from p03_bipartite_matching import max_matching

DIAMOND = [(0, 1, 10), (1, 3, 3), (0, 2, 5), (2, 3, 5)]
CLRS = [(0, 1, 16), (0, 2, 13), (1, 2, 10), (1, 3, 12), (2, 1, 4), (2, 4, 14), (3, 2, 9), (3, 5, 20), (4, 3, 7), (4, 5, 4)]

print("module 17: Network Flow and Matching")
print("problems available:", 6)
for name in ['p01_max_flow_value', 'p02_min_cut_edges', 'p03_bipartite_matching', 'p04_edge_disjoint_paths', 'p05_vertex_disjoint_paths', 'p06_assign_with_capacity']:
    print(f"  {name}")

## 1. Baseline — `p01_max_flow_value`

The first property, asserted rather than printed. Read the assertions before
running: each one names a specific input class, and most cross-check against an
independent brute force over the same data.

In [ ]:
assert max_flow_value(4, DIAMOND, 0, 3) == 8
assert max_flow_value(6, CLRS, 0, 5) == 23
# Bottleneck in the middle of a single chain.
assert max_flow_value(4, [(0, 1, 10), (1, 2, 3), (2, 3, 10)], 0, 3) == 3
# No route at all.
assert max_flow_value(4, [(0, 1, 5), (2, 3, 5)], 0, 3) == 0
assert max_flow_value(2, [], 0, 1) == 0
# Zero-capacity edges carry nothing.
assert max_flow_value(3, [(0, 1, 0), (1, 2, 5)], 0, 2) == 0
# The graph that needs flow to be taken back again.
reroute = [(0, 1, 1), (0, 2, 1), (1, 2, 1), (1, 3, 1), (2, 3, 1)]
assert max_flow_value(4, reroute, 0, 3) == 2
# Parallel edges add up.
assert max_flow_value(2, [(0, 1, 3), (0, 1, 4)], 0, 1) == 7

print("all assertions held")

## 2. Predict before you run

Five workers, five jobs, and a greedy assignment that takes each pair if both ends are free. Predict whether greedy can ever be beaten when every worker is qualified for exactly two jobs. Then predict what happens to the *count* of assignments if the source capacity is set to 2 by mistake - does it go up, down, or stay the same?

Commit to an answer before executing the next cell. Predicting and being wrong
is what makes the correction stick; reading the output first does not.

In [ ]:
# Several minimum cuts exist here, all worth 8. Residual exploration from
# the source yields this one - the source-minimal cut.
assert min_cut_edges(4, DIAMOND, 0, 3) == [(0, 2), (1, 3)]
assert sum(c for a, b, c in DIAMOND if (a, b) in [(0, 2), (1, 3)]) == 8

# The theorem: the cut's capacity equals the max flow, on random graphs.
random.seed(1702)
for _ in range(120):
    nodes = random.randint(2, 7)
    edges = [(a, b, random.randint(1, 6))
             for a in range(nodes) for b in range(nodes)
             if a != b and random.random() < 0.4]
    sink = nodes - 1
    flow = max_flow_value(nodes, edges, 0, sink)
    cut = min_cut_edges(nodes, edges, 0, sink)
    assert sum(c for a, b, c in edges if (a, b) in cut) == flow
    # And removing exactly those edges must disconnect the sink.
    survivors = [(a, b, c) for a, b, c in edges if (a, b) not in cut]
    assert max_flow_value(nodes, survivors, 0, sink) == 0

print("all assertions held")

## 3. Measurement

Claims about complexity are claims about wall-clock behaviour at scale, so they
have to be measured rather than asserted from the shape of the code.

In [ ]:
started = time.perf_counter()

assert max_matching(2, 2, [(0, 0), (0, 1), (1, 0)]) == 2
assert max_matching(3, 3, [(0, 0), (1, 1), (2, 2)]) == 3
assert max_matching(2, 2, []) == 0
assert max_matching(3, 1, [(0, 0), (1, 0), (2, 0)]) == 1
# Against an exhaustive search on small inputs.
random.seed(1703)
for _ in range(120):
    left = random.randint(1, 4)
    right = random.randint(1, 4)
    pairs = [(a, b) for a in range(left) for b in range(right)
             if random.random() < 0.5]
    best = 0
    for size in range(min(left, right), 0, -1):
        if any(len({a for a, _ in c}) == size and len({b for _, b in c}) == size
               for c in itertools.combinations(pairs, size)):
            best = size
            break
    assert max_matching(left, right, pairs) == best

elapsed = (time.perf_counter() - started) * 1000
print(f"all assertions held in {elapsed:.2f} ms")

## 4. Fix this cell

The values below are **wrong on purpose**. Run it, read the failure, work out
the right numbers from the cells above, and correct them in place.

Change only the expected values — not the code that computes them.

In [ ]:
import os

# DELIBERATELY BROKEN - two expected values, both wrong. Fix in place.

expected_problem_count = 99      # how many problems does this module ship?
expected_solution_count = 99     # how many reference solutions are on disk?

problem_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems") if f.startswith("p") and f.endswith(".py")
)
solution_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems" / "solutions")
    if f.startswith("p") and f.endswith(".py")
)

assert expected_problem_count == len(problem_files), (
    f"expected {expected_problem_count} problems, found {len(problem_files)}"
)
assert expected_solution_count == len(solution_files), (
    f"expected {expected_solution_count} solutions, found {len(solution_files)}"
)
print("Both match. Every problem has exactly one reference solution.")

## Takeaways

1. The residual edge is an accounting entry, not a road - it is what lets the algorithm change its mind.
2. Max flow equals min cut exactly - a free correctness check on your own code.
3. Almost every flow problem is a reduction; the work is building the graph.

### Where to go next

- [`01_README.md`](01_README.md) — the concepts in depth
- [`problems/README.md`](problems/README.md) — all 6 problems, with hint ladders
- [`debug_lab/SYMPTOMS.md`](debug_lab/SYMPTOMS.md) — planted defects that exit 0
- [Pattern Recognition Guide](../PATTERN_RECOGNITION_GUIDE.md) — attacking an unseen problem